In [1]:
!pip install rectools-lightfm catboost implicit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.7/411.7 kB 21.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00


In [6]:
!unzip /content/train_df.tsv.zip

Archive:  /content/train_df.tsv.zip
  inflating: train_df.tsv            
  inflating: __MACOSX/._train_df.tsv  


In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path

import lightfm
from lightfm.data import Dataset
from lightfm import LightFM

from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender, CosineRecommender
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [4]:
base_dir = Path('.')
train = pd.read_csv(base_dir / 'train_df.tsv', sep='\t').drop(columns=['Unnamed: 0'])
test_ids = pd.read_csv(base_dir / 'test_customer_ids.csv')
sample_submission = pd.read_csv(base_dir / 'sample_submission_7.csv')

In [5]:
train_community_ids = {v: k for k, v in enumerate(train.community_id.unique())}
train_customer_ids = {v: k for k, v in enumerate(train.customer_id.unique())}
inv_train_community_ids = {k: v for k, v in enumerate(train.community_id.unique())}
inv_train_customer_ids = {k: v for k, v in enumerate(train.customer_id.unique())}

train['community_id'] = train['community_id'].map(train_community_ids)
train['customer_id']  = train['customer_id'].map(train_customer_ids)

KeyboardInterrupt: 

In [ ]:
import pickle

with open('ids_maps', 'wb') as f:
  pickle.dump({
      'community_map': train_community_ids,
      'customer_map': train_customer_ids,
      'inv_community_map': inv_train_community_ids,
      'inv_customer_map': inv_train_customer_ids
  }, f)

In [ ]:
def fill_cat(data, cols):
  for col in cols:
    data[col] = data[col].astype('category').cat.add_categories(f'unknown_{col}')
    data[col] = train[col].fillna(f'unknown_{col}')

  data['description'] = data['description'].fillna('')
  data['join_request_date'] = data['join_request_date'].fillna(0).astype(int)
  return data

train = fill_cat(train, ['region_id', 'themeid', 'business_category', 'business_parent'])

In [10]:
encoder = LabelEncoder()
train['target'] = encoder.fit_transform(train['status'])
train = train.drop(columns=['status'])
train['target'] = train['target'].apply(lambda x: 0 if x == 0 else 1)

In [11]:
train['description'] = train['description'].apply(lambda x: 0 if x != '' else 1)
train.head()

,community_id,description,customers_count,messages_count,type,region_id,themeid,business_category,business_parent,customer_id,join_request_date,target
0,0,0,2966,1,7,unknown_region_id,unknown_themeid,unknown_business_category,unknown_business_parent,0,0,0
1,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,1,0,0
2,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,2,0,0
3,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,3,0,0
4,2,0,1035,1,7,10417791639.0,unknown_themeid,FAN_CLUB,BLOG,4,0,0


In [12]:
len(train_community_ids)

149114

## TFIDFRecommender

In [177]:
from tqdm.notebook import tqdm

rows = train.customer_id
cols = train.community_id

data_full = csr_matrix(
    ((np.ones(len(rows))), (rows, cols)),
    shape=(len(train_customer_ids), len(train_community_ids))
)

tfidf = TFIDFRecommender(K=7)
tfidf.fit(data_full)

popular = train.community_id.value_counts().index.to_list()
submission = []

for cid_real in tqdm(sample_submission.customer_id):
  cid = train_customer_ids.get(cid_real)
  if cid is None:
    recs = popular[:7]
  else:
    c_row = data_full[cid]
    preds = tfidf.recommend(cid, c_row)[0]
    recs = [i for i in preds[:7]]

  if len(recs) < 7:
    for pop in popular:
      if pop not in recs:
        recs.append(pop)
      if len(recs) == 7: break

  row = {'customer_id': cid_real} # Changed cid to cid_real to map back to original customer_id
  for i, r in enumerate(recs):
    row[f'community_id_{i+1}'] = inv_train_community_ids[r] # Mapped back to original community_id

  submission.append(row)

submission = pd.DataFrame(submission)
submission.to_csv('tfidf_submission.csv', index=False)

/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.08306288719177246 seconds
  warnings.warn(


  0%|          | 0/149114 [00:00<?, ?it/s]

  0%|          | 0/69046 [00:00<?, ?it/s]

In [178]:
submission.head()

,customer_id,community_id_1,community_id_2,community_id_3,community_id_4,community_id_5,community_id_6,community_id_7
0,d811a09d435ac2d1b1ed46e272405af10933b4711f4564...,e2d8710c3ca6ccb07ca3fbf66f0530be94903a58f5c2a8...,1379dd70dcdd575c69978f1b0b477d2f27222384313439...,caa892b683f011961b8868a73ce4999f667aa302d5a838...,6b5fcc67080b2a0359a15aac1b34a1f3e869211a887d0b...,ab5b19507c5a96c274a875f90d372882bde7956796df51...,768680a00db384987ff7532abc1ebc8062281b4ac60124...,95f663e12ea58f926d92f764b07674a53b29a5af1f5e2e...
1,73821118fc33500efaa6b1adf8ab0e9d314abb15f62603...,fc781628c86a7ee722f0efc9b75c9bf5a1360855a1e665...,d1af495ca20772cd9fcb43e8c27da695b5e9d93c64fc39...,53747ccd59fc17eac767624975cdd2de14aa6efa6090ae...,0380de4e396e8d4bd903d58299f67d06a84d9e8350250a...,c1ecc505d7c96ebbbc12129592f02928806dce4ade2023...,a437b788fc91efab69b24b19e18d88142667d0fc39c48a...,510a4b83bde73a7784dff4e1d89f67c943f4ff32b7f6b6...
2,6381971c002097a94b8d7a03d9dc3e9ff7872a52c4764a...,d42e10f2610272c545292b60844df2eb7a3ba47d6b4829...,9ecd44db17abc1fc665d38db272893274d5761665eb27c...,ce7a456428f786c44c3e27c039c4299885da5430ccfd05...,b51aa24cf528ab4b642b14241fad0ac42757dbfb97db53...,4bc61bb719c9fc971b43f8d563239653d5c8bbb1412016...,66cd88e7efdb90ca0107aa6808cea3b00df6dd44307349...,28ea66f6db0af1a3ed137ae38514d716e3124821f1731e...
3,250d49b476af0c7d2b23dd39cbb6edff39d44c64f8ebc0...,a8b15b543469eebe27c67c0c5897f6dc18e160cc5997c0...,892f2c17a6b9684c4dfd9834c0c6c8e7694c9faa00a0b3...,16183b4e07446d00c10d26deb9a1d62814d67fb6039b59...,96171649ea1057186a03c0f9bd216569642e431993897d...,2165f7bfb5b82ede0cf8f3be218c7f57ef2cd1e282ae9c...,a8d0b95bb7330d3a1171c439c5db88e1a3671edfbca950...,15382289ab5a19e157909ea41dad124e353009a3d1e883...
4,2da339a6bfe7791329ca8eb0a11544d4a9fb4c89572716...,3823a5e13dfd871d3ff72d5a69af9b77466d604f43cecd...,b8ed0e918d9f11140c70ef08be19c197917f4f4455f255...,aee65cc087f56024df2aedd0755db2a3d193cf2ca558b7...,4d7cf04c8e56f0797012064220d0a45509a1264bc17f58...,14d2fb073c9a2929f1fd9919cbbae50a5c597d2459d7f5...,1717cd8dbd989b4bd56009c53db75d564b252a3f3ed722...,b4823c708c5fdba713f8f150327ce2101a5061d4f9d1f7...


## LightFM

In [13]:
dataset = Dataset()

dataset.fit(
    users=train_customer_ids.values(),
    items=train_community_ids.values()
)

def get_features(data, drop: str):
  cols = [col for col in data.columns if col != drop]
  features = []
  for col in cols:
    vals = data[col].unique()
    for val in vals:
      features.append(f'{col}:{val}')

  return features

user_features = get_features(train, 'customer_id')
item_features = get_features(train, 'community_id')

dataset.fit_partial(
    user_features=user_features,
    item_features=item_features
)

In [14]:
lightfm_mapping = dataset.mapping()

lightfm_mapping = {
    'user_mapping': lightfm_mapping[0],
    'user_features_mapping': lightfm_mapping[1],
    'item_mapping': lightfm_mapping[2],
    'item_features_mapping': lightfm_mapping[3],
}

lightfm_mapping['user_inv_mapping'] = {v: k for k, v in lightfm_mapping['user_mapping'].items()}
lightfm_mapping['item_inv_mapping'] = {v: k for k, v in lightfm_mapping['item_mapping'].items()}

In [15]:
def df_to_tuple_iterator(df):
    return zip(*df.values.T)

def concat_last_to_list(t):
    return (t[0], list(t[1: ])[0])

def df_to_tuple_list_iterator(df):
    return map(concat_last_to_list, df_to_tuple_iterator(df))

In [16]:
train_mat, train_mat_weights = dataset.build_interactions(df_to_tuple_iterator(train[['customer_id', 'community_id', 'target']]))

In [17]:
train.head()

,community_id,description,customers_count,messages_count,type,region_id,themeid,business_category,business_parent,customer_id,join_request_date,target
0,0,0,2966,1,7,unknown_region_id,unknown_themeid,unknown_business_category,unknown_business_parent,0,0,0
1,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,1,0,0
2,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,2,0,0
3,1,1,8982,2,7,10424034685.0,246.0,unknown_business_category,unknown_business_parent,3,0,0
4,2,0,1035,1,7,10417791639.0,unknown_themeid,FAN_CLUB,BLOG,4,0,0


In [18]:
user_features = ['join_request_date']
item_features = ['description', 'customers_count', 'messages_count', 'type', 'region_id', 'themeid', 'business_category', 'business_parent']

user_types_map = {'join_request_date': int}
item_types_map = {'description': str,
                  'customers_count': int,
                  'messages_count': int,
                  'type': object,
                  'region_id': object,
                  'themeid': object,
                  'business_category': object,
                  'business_parent': object}

In [19]:
users = train[['customer_id'] + user_features]
users['features'] = users[user_features].apply(lambda x: [f'{col}:{x[col]}' for col in user_features], axis=1)

items = train[['community_id'] + item_features]
items['features'] = items[item_features].apply(lambda x: [f'{col}:{x[col]}' for col in item_features], axis=1)

/tmp/ipython-input-3778/1192018198.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  users['features'] = users[user_features].apply(lambda x: [f'{col}:{x[col]}' for col in user_features], axis=1)
/tmp/ipython-input-3778/1192018198.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items['features'] = items[item_features].apply(lambda x: [f'{col}:{x[col]}' for col in item_features], axis=1)


In [20]:
known_users = users['customer_id'].isin(train['customer_id'].unique())
train_user_features = dataset.build_user_features(
    df_to_tuple_iterator(users.loc[known_users, ['customer_id', 'features']])
)

known_items = items['community_id'].isin(train['community_id'].unique())
train_item_features = dataset.build_item_features(
    df_to_tuple_iterator(items.loc[known_items, ['community_id', 'features']])
)

In [ ]:
lfm = LightFM(
    loss='warp',
    no_components=32,
    learning_rate=0.05,
    random_state=42
)

num_epochs = 2
for epoch in tqdm(range(num_epochs)):
  lfm.fit_partial(
      train_mat,
      user_features=train_user_features,
      item_features=train_item_features,
      num_threads=2
  )

with open('lightfm_model.pkl', 'wb') as f:
    pickle.dump(lfm, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import sys
%%time

# time == 771m 40.5 s
def generate_lightfm_recs_batch(model, user_ids, item_ids, known_items,
                                 N, user_features, item_features,
                                 user_mapping, item_inv_mapping, num_threads=4):
    item_ids_arr = np.array(item_ids, dtype=np.int32)
    n_items = len(item_ids_arr)

    result = {}
    total = len(user_ids)

    for i, uid in enumerate(user_ids):
        if i % 500 == 0:
            print(f"\r[{i}/{total}] ({100*i/total:.1f}%)", end="", flush=True)

        if uid not in user_mapping:
            continue

        int_uid = user_mapping[uid]
        user_ids_repeated = np.full(n_items, int_uid, dtype=np.int32)

        scores = model.predict(
            user_ids_repeated,
            item_ids_arr,
            user_features=user_features,
            item_features=item_features,
            num_threads=num_threads
        )

        # Зануляем known items
        user_known = known_items.get(uid, [])
        if user_known:
            for vid in user_known:
                iid = lightfm_mapping['item_mapping'].get(vid)
                if iid is not None:
                    scores[iid] = -np.inf

        # argpartition быстрее полной сортировки
        top_idx = np.argpartition(-scores, N)[:N]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        recs = [item_inv_mapping[idx] for idx in top_idx if idx in item_inv_mapping]
        result[uid] = recs

    print(f"\r[{total}/{total}] (100.0%) — Done!")
    return result

known_items = train.groupby('user_id')['video_id'].apply(list).to_dict()
all_item_ids = list(lightfm_mapping['item_mapping'].values())

mapper = generate_lightfm_recs_batch(
    lfm,
    user_ids=train['customer_id'].values,
    item_ids=all_item_ids,
    known_items=known_items,
    N=100,
    user_features=train_user_features,
    item_features=train_item_features,
    user_mapping=lightfm_mapping['user_mapping'],
    item_inv_mapping=lightfm_mapping['item_inv_mapping'],
    num_threads=4
)

In [ ]:
lfm_sub = pd.DataFrame({
    'customer_id': sample_submission.customer_id.unique()
})

In [ ]:
lfm_sub['candidates'] = lfm_sub['customer_id'].map(mapper)
lfm_sub